In [107]:
# Loading the dataset
import requests
import torch

# Use MPS when available; otherwise fall back to CPU.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text.lower()

# Create a vocabulary
vocab = sorted(set(text))
vocab_size = len(vocab)

# Create mappings from characters to integers and vice versa
itos = {i: ch for i, ch in enumerate(vocab)}
stoi = {ch: i for i, ch in enumerate(vocab)}

# Convert data to ints
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

# Create train/test split
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

def get_train_data(batch_size, seq_length, stride=None):
    # Use strided windows for faster training set creation.
    if stride is None:
        stride = seq_length

    windows = train_data.unfold(0, seq_length + 1, stride)

    X = windows[:, :-1]
    y = windows[:, -1]

    # Trim data for full batches
    n = batch_size * (X.size(0) // batch_size)
    X = X[:n]
    y = y[:n]

    # Batch the data as (num_batches, batch_size, seq_length)
    X = X.view(-1, batch_size, seq_length)
    y = y.view(-1, batch_size)

    return X.to(device), y.to(device)

Xs, ys = get_train_data(batch_size=256, seq_length=64, stride=64)

In [ ]:
# Creating the PyTorch RNN model
import torch.nn as nn


class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(RNN, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.embedding =  nn.Embedding(self.vocab_size, self.embedding_dim)
        self.rnn = nn.RNN(self.embedding_dim, self.hidden_size, batch_first = True)
        self.linear = nn.Linear(self.hidden_size, self.vocab_size)

    def forward(self, Xs):
        return self.linear(self.rnn(self.embedding(Xs))[0])
    

In [117]:
# The model
model = RNN(
    vocab_size=vocab_size,
    embedding_dim=10,
    hidden_size=64,
).to(device)

# Optional compile can reduce Python overhead in the training step.
if hasattr(torch, "compile"):
    model = torch.compile(model)

In [125]:
# Training

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train(epochs=5):
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        # Keep permutation on the same device to avoid extra transfers.
        perm = torch.randperm(Xs.size(0), device=Xs.device)
        Xs_epoch = Xs[perm]
        ys_epoch = ys[perm]

        # Train one mini-batch at a time across all windows.
        for i in range(Xs_epoch.size(1)):
            X_batch = Xs_epoch[:, i, :]      # (num_windows, seq_length)
            y_batch = ys_epoch[:, i]         # (num_windows,)

            optimizer.zero_grad(set_to_none=True)
            logits = model(X_batch)[:, -1, :]  # Predict next char from last step.
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch} | Loss: {total_loss / Xs_epoch.size(1):.4f}")

In [129]:
train()



Epoch 0 | Loss: 1.6369
Epoch 1 | Loss: 1.6319
Epoch 2 | Loss: 1.6265
Epoch 3 | Loss: 1.6212
Epoch 4 | Loss: 1.6163


In [130]:
train()

Epoch 0 | Loss: 1.6112
Epoch 1 | Loss: 1.6066
Epoch 2 | Loss: 1.6024
Epoch 3 | Loss: 1.5982
Epoch 4 | Loss: 1.5939


In [131]:
# Generate text from a seed taken from test_data

def generate_from_test(model, test_data, start_idx=0, seed_len=40, gen_len=200, temperature=0.9):
    model.eval()

    # Build seed from the test split
    seed_ids = test_data[start_idx : start_idx + seed_len].tolist()
    x = torch.tensor(seed_ids, dtype=torch.long, device=device).unsqueeze(0)  # (1, T)

    with torch.no_grad():
        for _ in range(gen_len):
            logits = model(x)[:, -1, :] / max(temperature, 1e-6)  # (1, vocab_size)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)     # (1, 1)
            x = torch.cat([x, next_id], dim=1)

    seed_text = "".join(itos[i] for i in seed_ids)
    full_text = "".join(itos[i] for i in x[0].tolist())
    return seed_text, full_text


# Example
seed_text, generated_text = generate_from_test(
    model,
    test_data,
    start_idx=0,
    seed_len=40,
    gen_len=300,
    temperature=0.8,
)

print("SEED:")
print(seed_text)
print("\nGENERATED:")
print(generated_text)

SEED:
?

gremio:
good morrow, neighbour baptis

GENERATED:
?

gremio:
good morrow, neighbour baptis:
why with more he protcest:
where'gs, but you han.

madjood far sume endiion'd yee, theman:
come.

morpoon ule of defeing this spairt, that yere him soude is awen gimer:
i knreld:
they pryicess nems, and congs?

lorded my pryes, cim even of of murder them.

bursink it?
bath' the peather wratfolt co
